# Домашнє завдання: метрики класифікації на Iris

На парі ми писали **KNN** — алгоритм, який шукає найближчі квітки й визначає клас за голосуванням сусідів. **KNN уже готовий нижче**: запустіть клітинки, прочитайте коментарі та розберіться, як він працює.

**Ваше домашнє завдання — самостійно порахувати метрики**, без `sklearn.metrics`. Достатньо циклів `for`, `if`, списків і звичайної арифметики.


## 1. Завантажуємо дані

У Iris є 150 квіток трьох видів (класи `0`, `1`, `2`). Кожна квітка описана **4 числами**: довжиною та шириною чашолистка й пелюстки.

`train_data` — приклади, з якими порівнюємо нову квітку; `test_data` — квітки, для яких робимо прогноз. `test_labels` — правильні відповіді, потрібні лише для перевірки.


In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split

# Завантажуємо справжній набір даних Iris.
iris = datasets.load_iris()

# 80% даних залишаємо для навчання, 20% — для перевірки.
# Функція повертає спочатку train, а потім test.
train_data, test_data, train_labels, test_labels = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

print("Навчальних квіток:", len(train_data))
print("Тестових квіток:", len(test_data))
print("Назви класів:", iris.target_names)


## 2. Готовий KNN — розбираємо код із дошки

Для кожної тестової квітки:

1. Рахуємо відстань до **кожної** навчальної квітки:

   $$d=\sqrt{(x_1-y_1)^2+(x_2-y_2)^2+(x_3-y_3)^2+(x_4-y_4)^2}$$
2. Сортуємо всі відстані від найменшої до найбільшої.
3. Беремо `k = 5` найближчих квіток.
4. Рахуємо, скільки серед цих сусідів квіток класу `0`, `1` і `2`.
5. Обираємо **найчастіший клас** — це наш прогноз.

Усі ці кроки вже написані. **Нічого дописувати в KNN не потрібно.**


In [ ]:
k = 5  # Скільки найближчих сусідів беремо для голосування.
predictions = []  # Сюди складатимемо передбачений клас кожної тестової квітки.

# Беремо тестові квітки по одній.
for test_item in test_data:
    distances = []  # Відстані від цієї квітки до всіх навчальних квіток.

    # Порівнюємо поточну тестову квітку з кожною навчальною.
    for train_item in train_data:
        distance_squared = 0  # Сума квадратів різниць спочатку дорівнює нулю.

        # У кожної квітки 4 ознаки. Порівнюємо відповідні числа.
        for x, y in zip(test_item, train_item):
            distance_squared += (x - y) ** 2  # Додаємо квадрат різниці.

        distance = distance_squared ** 0.5  # Беремо квадратний корінь.
        distances.append(distance)  # Зберігаємо відстань.

    # Поєднуємо кожну відстань із класом відповідної навчальної квітки.
    # Сортуємо пари: найменша відстань буде першою.
    sorted_neighbors = sorted(zip(distances, train_labels))

    # Беремо тільки перших 5 (k) найближчих сусідів.
    nearest = sorted_neighbors[:k]

    # Три лічильники: скільки сусідів належить до кожного класу.
    count0 = 0
    count1 = 0
    count2 = 0

    # У кожній парі: distance — відстань, label — клас сусіда.
    for distance, label in nearest:
        if label == 0:
            count0 += 1
        elif label == 1:
            count1 += 1
        else:
            count2 += 1

    # Шукаємо клас, який зустрівся найчастіше.
    # Якщо є рівність, залишаємо клас із меншим номером.
    predicted_class = 0
    biggest_count = count0

    if count1 > biggest_count:
        biggest_count = count1
        predicted_class = 1

    if count2 > biggest_count:
        biggest_count = count2
        predicted_class = 2

    # Зберігаємо один прогноз для поточної тестової квітки.
    predictions.append(predicted_class)

# Дивимося, що передбачила модель і які класи були правильними.
print("Прогнози:       ", predictions)
print("Правильні класи:", list(test_labels))


## 3. Домашнє завдання: метрики своїми руками

**Не використовуйте `sklearn.metrics`** і готові функції для метрик. Пишіть простими циклами `for`, умовами `if` та арифметикою. Усі потрібні дані вже є: `predictions` і `test_labels`.

### Завдання 1. Accuracy

**Accuracy** показує, яка частка всіх передбачень правильна.

$$Accuracy = \frac{\text{кількість правильних прогнозів}}{\text{кількість усіх прогнозів}}$$

Підказка: пройдіться циклом по індексах від `0` до `len(test_labels) - 1`. Якщо `predictions[i] == test_labels[i]`, збільште лічильник `correct` на 1. Потім поділіть його на кількість тестових квіток.


In [ ]:
# ЗАВДАННЯ 1: порахуйте Accuracy.
correct = 0  # Скільки прогнозів збіглося з правильною відповіддю.

for i in range(len(test_labels)):
    if predictions[i] == test_labels[i]:
        correct += 1

accuracy = correct / len(test_labels)
print("Правильних прогнозів:", correct, "з", len(test_labels))
print("Accuracy:", accuracy)

### Завдання 2. TP, FP, FN, TN для кожного класу

У нас **3 класи**, тому перевіряємо кожен клас окремо: спочатку `0`, потім `1`, потім `2`.

Уявіть, що зараз нас цікавить тільки **клас 0**. Тоді:

- **TP (True Positive)** — насправді клас `0`, і модель сказала `0`.
- **FP (False Positive)** — насправді **не** `0`, але модель сказала `0`.
- **FN (False Negative)** — насправді `0`, але модель сказала **не** `0`.
- **TN (True Negative)** — насправді **не** `0`, і модель сказала **не** `0`.

Для класів `1` і `2` логіка точно така сама.

**Завдання:** для кожного класу порахуйте чотири лічильники. Підказка: зовнішній цикл `for class_id in range(3)`, внутрішній — по всіх тестових квітках. Усередині перевіряйте `test_labels[i]` та `predictions[i]` через `if` / `elif` / `else`.

Перевірка себе: **TP + FP + FN + TN = кількість усіх тестових квіток** для кожного класу.


In [ ]:
# ЗАВДАННЯ 2 і 3: TP, FP, FN, TN, Precision, Recall, F1 для кожного класу.
for class_id in range(3):
    tp = 0
    fp = 0
    fn = 0
    tn = 0

    for i in range(len(test_labels)):
        actual = test_labels[i]
        predicted = predictions[i]

        if actual == class_id and predicted == class_id:
            tp += 1
        elif actual != class_id and predicted == class_id:
            fp += 1
        elif actual == class_id and predicted != class_id:
            fn += 1
        else:
            tn += 1

    print("Клас:", class_id, "TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
    print("Перевірка суми:", tp + fp + fn + tn == len(test_labels))

    if tp + fp == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)

    if tp + fn == 0:
        recall = 0
    else:
        recall = tp / (tp + fn)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    print("Precision:", precision, "Recall:", recall, "F1:", f1)
    print()

### Завдання 3. Precision, Recall та F1-score

Використайте `TP`, `FP`, `FN` для кожного класу з попереднього завдання.

**Precision** — серед квіток, які модель назвала цим класом, яка частка справді належить до нього?

$$Precision = \frac{TP}{TP+FP}$$

**Recall** — серед усіх справжніх квіток цього класу яку частку модель знайшла?

$$Recall = \frac{TP}{TP+FN}$$

**F1-score** — число, яке враховує і Precision, і Recall:

$$F1 = \frac{2\cdot Precision\cdot Recall}{Precision+Recall}$$

**Завдання:** порахуйте три метрики **окремо для класів 0, 1, 2**. Найпростіше — дописати обчислення **всередині циклу із завдання 2**, одразу після підрахунку TP/FP/FN/TN.

Якщо знаменник дорівнює нулю, для цього завдання вважайте результат `0` (перед діленням перевіряйте `if`).


In [ ]:
# ЗАВДАННЯ 3 виконано в клітинці вище (завдання 2): Precision, Recall і F1
# рахуються всередині того самого циклу одразу після TP, FP, FN, TN.

### Короткі питання (відповідь своїми словами)

1. Чим **Precision** відрізняється від **Recall**?
2. Чи може бути висока **Accuracy**, але поганий **Recall** для одного класу? Чому?
3. Для якого з трьох класів ваш KNN працює найкраще за **F1-score**?

**Що здати:** цей ноутбук із заповненими завданнями 1–3 та короткими відповідями. KNN змінювати не треба.


In [ ]:
# Відповіді на питання:
# 1. Precision показує, яка частка квіток, названих моделлю певним класом, справді
#    належить до цього класу (TP / (TP + FP)); вона штрафує хибні спрацювання (FP).
#    Recall показує, яку частку всіх справжніх квіток цього класу модель знайшла
#    (TP / (TP + FN)); він штрафує пропущені квітки (FN).
# 2. Так, може. Accuracy рахується по всіх класах разом, тому якщо клас рідкісний,
#    модель може майже завжди помилятися на ньому (низький Recall), але правильно
#    передбачати решту, і загальна Accuracy залишиться високою.
# 3. На цьому тестовому наборі для всіх трьох класів F1 = 1.0 (усі 30 прогнозів
#    правильні), тому класи однаково хороші; найкращого одного немає.